In [27]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from collections import Counter

# list storing types, will be our keys for dicts keeping track of positives and negatives
pokemon_types = ["normal", "fire", "water", "grass", 
                 "electric", "ice", "fighting", "poison",
                 "ground", "flying", "psychic", "bug",
                 "rock", "ghost", "dragon", "dark",
                 "steel", "fairy"]

with open("pokemon_data.json", "r") as file:
    data = json.load(file)
data_df = pd.DataFrame(data)

# shuffling the data set by sampling all of it with a set random seed
data_df = data_df.sample(frac = 1, random_state = 727)
indices = np.array_split(np.arange(len(data_df)), 20)
split_df = [data_df.iloc[idx] for idx in indices]
# splitting 70% of the data into training, 15% each into testing and validation
training_set = pd.concat(split_df[0:14])
testing_set = pd.concat(split_df[14:17])
validation_set = pd.concat(split_df[17:])
# the combined set will be used for testing for algorithms that do not require validation
combined_set = pd.concat(split_df[14:])

In [28]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

stats = ["hp", "attack", "defense", "special-attack", "special-defense", "speed"]

training_set[stats] = scaler.fit_transform(training_set[stats])
validation_set[stats] = scaler.transform(validation_set[stats])
testing_set[stats] = scaler.transform(testing_set[stats])

In [29]:
# double checking to ensure each set has the right number of pokemon
# can see here the training set has about 700 pokemon, testing and validation set each have about 150
print(training_set.shape)
print(testing_set.shape)
print(validation_set.shape)

(738, 11)
(156, 11)
(156, 11)


In [30]:
# helper function to determine precision, recall, and F1 score for each benchmark
def performance_metrics(true_positives, false_positives, true_negatives, false_negatives):
    precisions = dict.fromkeys(pokemon_types)
    recalls = dict.fromkeys(pokemon_types)
    f1scores = dict.fromkeys(pokemon_types)
    sum_precision = 0
    sum_recall = 0
    sum_f1 = 0

    for pokemon_type in pokemon_types:
        precisions[pokemon_type] = (true_positives[pokemon_type]) / (false_positives[pokemon_type] + true_positives[pokemon_type])
        sum_precision += precisions[pokemon_type]
        recalls[pokemon_type] = (true_positives[pokemon_type]) / (false_negatives[pokemon_type] + true_positives[pokemon_type])
        sum_recall += recalls[pokemon_type]
        f1scores[pokemon_type] = (2 * true_positives[pokemon_type]) / ( (2 * true_positives[pokemon_type]) + false_positives[pokemon_type] + false_negatives[pokemon_type])
        sum_f1 += f1scores[pokemon_type]

    return sum_precision/len(pokemon_types), sum_recall/len(pokemon_types), sum_f1/len(pokemon_types)

In [31]:
# helper function to mutate dicts keeping track of true/false positives/negatives
def mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, actual_types, predicted_type):
    if predicted_type in actual_types:
        for pokemon_type in pokemon_types:
            # if we get the type right,
            # it counts as a true positive for each type of the pokemon,
            # and as a true negative for every other type
            if pokemon_type in actual_types:
                true_positives[pokemon_type] += 1
            else:
                true_negatives[pokemon_type] += 1

    else:
        for pokemon_type in pokemon_types:
            # if we don't get the type right,
            # it counts as a false positive for the predicted type,
            # and a false negative for the types of the pokemon
            if pokemon_type in actual_types:
                false_negatives[pokemon_type] += 1
            elif pokemon_type == predicted_type:
                false_positives[pokemon_type] += 1
    

In [32]:
# majority class is water type
def majority_class_benchmark(testing_set, testing_size):
    true_positives = dict.fromkeys(pokemon_types, 0)
    false_positives = dict.fromkeys(pokemon_types, 0)
    true_negatives = dict.fromkeys(pokemon_types, 0)
    false_negatives = dict.fromkeys(pokemon_types, 0)

    majority_class = "water"
    num_correct = 0
    for index, row in testing_set.iterrows():
        if majority_class in row["types"]:
            num_correct += 1
        mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, row["types"], majority_class)

    for pokemon_type in pokemon_types:
        # this adjustment ensures we don't divide by 0 when we do our precision and recall calculations
        if false_positives[pokemon_type] == 0:
            false_positives[pokemon_type] += 1

    metrics = dict.fromkeys(["accuracy", "precision", "recall", "f1score"])
    metrics["accuracy"] = num_correct/testing_size
    metrics["precision"], metrics["recall"], metrics["f1score"] = performance_metrics(true_positives, false_positives, true_negatives, false_negatives)
    return metrics

In [33]:
# helper function to determine Euclidean distance between two pokemon based on base stats
def distance(pokemon1, pokemon2):
    hp = (pokemon1["hp"] - pokemon2["hp"]) ** 2
    attack = (pokemon1["attack"] - pokemon2["attack"]) ** 2
    defense = (pokemon1["defense"] - pokemon2["defense"]) ** 2
    special_attack = (pokemon1["special-attack"] - pokemon2["special-attack"]) ** 2
    special_defense = (pokemon1["special-defense"] - pokemon2["special-defense"]) ** 2
    speed = (pokemon1["speed"] - pokemon2["speed"]) ** 2
    return math.sqrt(hp + attack + defense + special_attack + special_defense + speed)

In [34]:
def nearest_neighbor_benchmark(training_set, testing_set, testing_size, k, training):
    true_positives = dict.fromkeys(pokemon_types, 0)
    false_positives = dict.fromkeys(pokemon_types, 0)
    true_negatives = dict.fromkeys(pokemon_types, 0)
    false_negatives = dict.fromkeys(pokemon_types, 0)

    num_correct = 0
    prediction = ""

    # iterating through each pokemon in our testing set
    for index1, row1 in testing_set.iterrows():
        distance_list = []

        # iterating through each pokemon in our training set
        for index2, row2 in training_set.iterrows():
            #if the training set is being used, skip itself in the set to not artificially inflate accuracy
            if training and index1 == index2:
                continue

            current_distance = distance(row1, row2)

            distance_list.append((current_distance, row2["types"]))
            
        sorted_distance_list = sorted(distance_list)
        k_nearest_neighbors = sorted_distance_list[:k] #sorts the distances and adds them k smallest distances to a new list

        types = [type for neighbor in k_nearest_neighbors for type in neighbor[1]]
        prediction = Counter(types).most_common(1)[0][0] #finds most common type

        if prediction in row1["types"]:
            num_correct += 1

        mutate_dicts(true_positives, false_positives, true_negatives, false_negatives, row1["types"], prediction)
        
    metrics = dict.fromkeys(["accuracy", "precision", "recall", "f1score"])
    metrics["accuracy"] = num_correct/testing_size
    metrics["precision"], metrics["recall"], metrics["f1score"] = performance_metrics(true_positives, false_positives, true_negatives, false_negatives)
    return metrics
        

In [35]:
print(f"{majority_class_benchmark(training_set, training_set.shape[0])}")
print(f"{majority_class_benchmark(validation_set, validation_set.shape[0])}")
print(f"{majority_class_benchmark(testing_set, testing_set.shape[0])}")

{'accuracy': 0.13956639566395665, 'precision': 0.6125596851206606, 'recall': 0.10812856660935612, 'f1score': 0.10853337266040096}
{'accuracy': 0.14102564102564102, 'precision': 0.28561253561253563, 'recall': 0.11274281274281275, 'f1score': 0.10679922176078296}
{'accuracy': 0.23076923076923078, 'precision': 0.3396723646723647, 'recall': 0.14847924406747934, 'f1score': 0.16118903026797762}


In [36]:
# training_f1score = []
# validation_f1score = []

# #tests loss of different k values from 1 to 15
# for k in range(1, 15):
#     training_results = nearest_neighbor_benchmark(training_set, training_set, training_set.shape[0], k, True)
#     validation_results = nearest_neighbor_benchmark(training_set, validation_set, validation_set.shape[0], k, False)

#     training_f1score.append(training_results["f1score"])
#     validation_f1score.append(validation_results["f1score"])

In [37]:
#plots graph of loss comparison between training and validation set
# plt.figure(figsize=(10, 6))

# plt.plot(range(1, 15), training_f1score, label="Training F1 Score", marker="o")
# plt.plot(range(1, 15), validation_f1score, label="Validation F1 Score", marker="o")

# plt.xlabel("K Value")
# plt.ylabel("F1 Score")
# plt.title("Training vs Validation F1 Score")

# plt.legend()
# plt.grid()
# plt.show()

In [38]:
print(f"{nearest_neighbor_benchmark(training_set, training_set, training_set.shape[0], 6, True)}")
print(f"{nearest_neighbor_benchmark(training_set, validation_set, validation_set.shape[0], 6, False)}")
print(f"{nearest_neighbor_benchmark(training_set, testing_set, testing_set.shape[0], 6, False)}")


{'accuracy': 0.22628726287262874, 'precision': 0.3390803794217582, 'recall': 0.22638095429421312, 'f1score': 0.2656191698158554}
{'accuracy': 0.3269230769230769, 'precision': 0.43036228716053276, 'recall': 0.3023232849213241, 'f1score': 0.3441231248596986}
{'accuracy': 0.22435897435897437, 'precision': 0.35178355595022265, 'recall': 0.22704816516091025, 'f1score': 0.2559729584661976}
